In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from statsmodels.tsa.arima_model import ARIMA
import pmdarima as pm
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import r2_score
from scipy import stats
from scipy.special import inv_boxcox

In [2]:
data = pd.read_excel('Forecast_demand_data.xlsx', sheet_name = 'vaccine_demand')

In [3]:
data = data.T

# Resetting the index so the first row becomes the header
data.reset_index(inplace=True)
new_header = data.iloc[0]  # Grab the first row for the header
data = data[1:]  # Take the data less the header row
data.columns = new_header  # Set the header row as the df header

data

,Vaccine,Measles,MR,MMR,TT,Td,DT,DTwP,DTwP-Hib,DTwP-HepB-Hib,IPV,OPV,HPV,HepB,Hib,Rota,PCV
1,2000,78580259.047222,16708549.179678,14895546.665213,78159428.700842,50764265.476615,20923950.020323,311598814.061627,4569986.526316,5598384.238158,0.0,301694795.29,0.0,20888757.517544,804259.789474,0.0,0.0
2,2001,81681321.359731,16708884.350395,15916936.759832,79754932.221155,50774337.539247,20532693.853787,314913212.590571,2613786.034079,11321282.009342,0.0,325020669.986447,0.0,23446088.279386,814409.567368,0.0,0.0
3,2002,79266752.609852,16621239.04136,16440302.312529,82572071.930308,50335231.889878,20111574.529955,275098221.465127,2314794.741316,21742186.647105,0.0,330516182.045092,0.0,67753162.6711,791281.727895,0.0,0.0
4,2003,86623864.744981,16912298.703333,17051794.304503,85747371.100019,52856121.304121,19392670.084335,262871459.777828,4993056.554211,23354085.385395,0.0,345227358.555789,0.0,74823326.876657,745679.866053,0.0,0.0
5,2004,86152078.769785,17990792.786474,25753121.47349,88823501.799947,69969918.250374,19054703.122363,261128636.204495,6731549.577763,34313913.773947,0.0,370122959.802895,0.0,82156425.247354,797608.277763,0.0,0.0
6,2005,90148923.016216,32224595.05761,26516485.200318,90980825.641052,68257614.967451,18693917.721384,257038004.931873,5829500.588289,44391889.895395,0.0,379929193.846184,0.0,85718419.53492,793505.360132,0.0,0.0
7,2006,91416944.607556,36010295.660687,28053836.320973,93434224.750214,54723842.706776,20305661.865547,243336949.277651,2388879.168553,49168454.713947,538568.8,376577517.66,0.0,83892402.907327,807322.461842,3029895.873684,0.0
8,2007,95425542.39019,35638478.889781,27963867.296272,98105257.648447,53185019.472065,19916221.879374,227800791.459953,2682333.997632,61360525.330132,572237.68,382669960.60579,0.0,85888544.022034,15497985.037632,7758595.657842,0.0
9,2008,120394966.339357,21093263.448026,39454388.755779,101078445.764158,38358186.250233,19937955.877737,225945393.957785,5503855.038947,68262550.020395,1564054.075,381567905.880263,9436.631579,102825622.134888,23015253.727711,1545856.413623,6300.631579
10,2009,110179825.434151,22353124.596009,41213108.991876,104582797.036569,36289140.125532,19825403.520702,232813533.268769,6149962.554342,121877662.134079,1802422.5725,395904555.623289,15032.895439,115113933.700781,29554012.882237,4340799.676158,1716887.879923


In [4]:
def safe_log(x, p):
    """
    Applies a transformation to a given pandas Series, leaving zero values unchanged and handling non-numeric types.

    Parameters:
    - x: A pandas Series.
    - p: Type of transformation ('log', 'log10', or any numeric value for power transformation).

    Returns:
    - A transformed pandas Series where zeros remain zeros.
    """
    # Convert x to a numeric type, coercing errors to NaN (which won't affect zeros)
    x_numeric = pd.to_numeric(x, errors='coerce')
    
    # Creating a mask for non-zero values
    mask = x_numeric != 0

    # Copy the original numeric series to preserve zeros and NaNs
    result = x_numeric.copy()

    # Apply transformations only to non-zero values
    if p == 'log':
        result[mask] = np.log(x_numeric[mask])
    elif p == 'log10':
        result[mask] = np.log10(x_numeric[mask])
    else:
        # For power transformations, assuming 'p' is a number
        result[mask] = np.power(x_numeric[mask], p)

    return result



    

def safe_return(x, p):
    """
    Reverses the inverse transformation for a given pandas Series.

    Parameters:
    - x: A pandas Series with values that have been log-transformed.
    - p: type of transformation

    Returns:
    - A pandas Series with the original values before the transformation.
    """
    if p == 'log':
        return np.exp(x)
    elif p == 'log10':
        return 10 ** x
    else:
        return np.power(x,1/p)


# def safe_log(x, p):
#     """
#     Reverses the transformation for a given pandas Series.

#     Parameters:
#     - x: A pandas Series.
#     - p: type of transformation

#     Returns:
#     - A transformed pandas Series.
#     """
#     if p == 'log':
#         return np.log(x)
#     elif p == 'log10':
#         return np.log10(x)
#     else:
#         return np.power(x,p)
    

# def safe_return(x, p):
#     """
#     Reverses the inverse transformation for a given pandas Series.

#     Parameters:
#     - x: A pandas Series with values that have been log-transformed.
#     - p: type of transformation

#     Returns:
#     - A pandas Series with the original values before the transformation.
#     """
#     if p == 'log':
#         return np.exp(x)
#     elif p == 'log10':
#         return 10 ** x
#     else:
#         return np.power(x,1/p)



In [5]:
def remove_leading_zeros(series):
    """
    Removes leading zeros from a pandas Series and returns the trimmed Series.
    """
    # Convert series to boolean where True indicates non-zero values
    non_zero_mask = series != 0
    
    # Find the index of the first non-zero value
    first_non_zero_index = non_zero_mask.idxmax()
    
    # Slice the series from the first non-zero value onwards
    trimmed_series = series.loc[first_non_zero_index:]
    
    return trimmed_series


In [6]:
def process_column_with_forecast(column_data, column_name, p):
    column_data = remove_leading_zeros(column_data).astype(float)
    # column_data = column_data.apply(safe_log)
    # transformed_data, lambda_value = stats.boxcox(column_data)
    # column_data = column_data / 1000000
    column_data = safe_log(column_data, p)
    # print(lambda_value)
    seasonal = False
    model = pm.auto_arima(column_data, 
                          m=1, seasonal=seasonal, d=None, test='adf', 
                          start_p=2, start_q=0, max_p=3, max_q=3, D=None,
                          max_order = 10, information_criterion = 'aic',
                          trace=False, error_action='ignore',  
                          suppress_warnings=True, stepwise=True)
    curr_model = model.order
    model.plot_diagnostics()
    plt.show()
    # plt.savefig(f'figs/{column_name}_diagnostic_plot.png')
    plt.close()

    # Forecasting
    fc, confint = model.predict(n_periods=10, return_conf_int=True)
    fitted_values = model.predict_in_sample()
    # print(fitted_values)

    # Calculating AIC value
    aic_value = model.aic()

    # Creating a DataFrame for the fitted and forecasted values
    fitted_values = safe_return(fitted_values, p)
    fitted_df = pd.DataFrame(fitted_values)
    fitted_df.rename(columns={'predicted_mean': f'{column_name}_fitted'}, inplace=True)
    # fitted_df.rename(columns={0: f'{column_name}_fitted'}, inplace=True)
    # print(fitted_df)
    
    fc = safe_return(fc, p)
    # fc = inv_boxcox(fc, lambda_value)
    fc_df = pd.DataFrame(fc, columns=[f'{column_name}_forecast'])
    
    # confint = inv_boxcox(confint, lambda_value)
    confint = safe_return(confint,p)
    confidence_df = pd.DataFrame(confint, columns=['Lower', 'Upper'])
    
    #testing changing indices
    if isinstance(data[column].index, pd.DatetimeIndex):
        # If your index is datetime, generate new dates that follow the last date of the historical data
        last_date = data[column].index[-1]
        forecast_dates = pd.date_range(start=last_date, periods=len(fc) + 1, freq='M')[1:]  # Adjust 'freq' as needed
    else:
        # For numeric indices, continue from the last index
        start = data[column].index[-1] + 1
        end = start + len(fc)
        forecast_dates = range(start, end)

    # Assign this new index to fc_df and confidence_df
    fc_df.index = forecast_dates
    confidence_df.index = forecast_dates
    
    print(f"Best Model: ARIMA{model.order}")

    # Adjust the index for the forecasted values and confidence intervals
    # fc_index = pd.RangeIndex(start=len(column_data), stop=len(column_data) + len(fc), step=1)
    # fc_df.index = fc_index
    # confidence_df.index = fc_index
    plt.figure(figsize=(10, 6))
    
    # Plotting historical data
    column_data = safe_return(column_data, p)
    column_data.plot(label='Historical')
    
    # Plotting forecasted values
    plt.plot(fc_df.index, fc_df[f'{column_name}_forecast'], color='red', label='Forecast')
    # Assuming 'fitted_values' is a DataFrame with the same index as 'data'
    # plt.plot(fitted_df.index, fitted_df[f'{column_name}_fitted'], color='green', label='Fitted')
    # Assuming 'confidence_df' is your DataFrame with forecast confidence intervals
    plt.fill_between(confidence_df.index, confidence_df['Lower'], confidence_df['Upper'], color='pink', alpha=0.3)
    plt.legend()
    plt.title(f'ARIMA Forecast with Confidence Intervals for {column_name}')
    plt.xlabel('Time')
    plt.ylabel('Values')
    # plt.text(0.05, 0.95, f'AIC = {aic_value:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle="round", alpha=0.5, facecolor='white'))
    plt.text(0.05, 0.95, f'AIC = {aic_value:.3f}\nModel = {curr_model}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle="round", alpha=0.5, facecolor='white'))
    # plt.savefig(f'figs/{column_name}_forecast_plot.png')  # Save plot to PNG
    plt.show()
    plt.close()

    
    # return fitted_df, fc_df, confidence_df, aic_value
    return confidence_df, aic_value

In [ ]:
# Assuming 'data' is your DataFrame with the actual data

# fitted_dfs = {}
# forecast_dfs = {}
confidence_dfs = {}
# AIC_scores = {}
exclude_columns = ['Measles', 'PCV', 'OPV', 'HPV', 'IPV', 'DTwP-HepB-Hib', 'DTwP', 'MR', 'MMR', 'Rota', 'TT', 'Td', 'DT']
df_selected = data.drop(columns=exclude_columns)

for column in df_selected.columns[1:]:
    print(column)
    df_confidence = process_column_with_forecast(data[column], column, 1)
    # df_fitted, df_forecast, df_confidence, aic_value = process_column_with_forecast(data[column], column)
    # fitted_dfs[column] = df_fitted
    # forecast_dfs[column] = df_forecast
    confidence_dfs[column] = df_confidence
    # AIC_scores[column] = aic_value


In [ ]:
confidence_dfs = {}
AIC_scores = {}
exclude_columns = []
df_selected = data.drop(columns=exclude_columns)

for column in df_selected.columns[1:]:
    for power in np.arange(0.01,0.1, 0.005):
        df_confidence, aic_value = process_column_with_forecast(data[column], column, power)
        confidence_dfs[column] = df_confidence
        AIC_scores[power] = aic_value

In [ ]:
excel_path = 'antigen_CIs.xlsx'  # Specify your file path here

with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a') as writer:
    # Iterate over the dictionary and write each DataFrame to a different sheet
    for sheet_name, df in confidence_dfs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
# Assuming 'data' is your DataFrame with the actual data

# fitted_dfs = {}
# forecast_dfs = {}
confidence_dfs = {}
# AIC_scores = {}
# exclude_columns = ['Diphtheria', 'Pertussis', 'Polio', 'Tetanus']
exclude_columns = ['Hib','HPV']
# df_selected = data.drop(columns=exclude_columns)

for column in data.loc[:, exclude_columns]:
    df_confidence = process_column_with_forecast(data[column], column, p=1)
    # df_fitted, df_forecast, df_confidence, aic_value = process_column_with_forecast(data[column], column)
    # fitted_dfs[column] = df_fitted
    # forecast_dfs[column] = df_forecast
    confidence_dfs[column] = df_confidence
    # AIC_scores[column] = aic_value

In [ ]:
confidence_dfs